In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import wandb
import random
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix, classification_report
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Bidirectional, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical


# --- SET SEEDS FOR REPRODUCIBILITY ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- WANDB INITIALIZATION ---
wandb.init(
    project="air-quality-prediction",
    name="aqi-classification-3class",
    config={
        "seed": SEED,
        "test_split_date": "2022-01-01",
        "lstm_epochs": 100,
        "lstm_batch_size": 32,
        "lstm_learning_rate": 0.001,
        "lstm_layers": [128, 64],
        "lstm_window_size": 72,
        "validation_split": 0.1,
        "target": "AQI_category",
        "num_classes": 3,
    }
)

# --- CONFIGURATION ---
FILE_PATH_LINEAR = "/content/drive/MyDrive/Project-master/data/Clean For Model/Air_Quality_linear_ready.csv"
FILE_PATH_LSTM = "/content/drive/MyDrive/Project-master/data/Clean For Model/Air_Quality_lstm_ready.csv"
TEST_SPLIT_DATE = pd.Timestamp('2022-01-01')
LSTM_WINDOW_SIZE = 72
NUM_CLASSES = 3  # Good, Moderate, Poor

target_col = 'target_category'

# --- SLIDING WINDOW FUNCTION ---
def create_sliding_windows(X, y, window_size):
    X_windows, y_windows = [], []
    for i in range(len(X) - window_size):
        X_windows.append(X[i:i + window_size])
        y_windows.append(y[i + window_size])
    return np.array(X_windows), np.array(y_windows)

# 1. DATA PREPARATION
df_linear = pd.read_csv(FILE_PATH_LINEAR)
df_lstm = pd.read_csv(FILE_PATH_LSTM)

df_linear['datetime'] = pd.to_datetime(df_linear['datetime'])
df_lstm['datetime'] = pd.to_datetime(df_lstm['datetime'])

df_linear = df_linear.set_index('datetime').sort_index()
df_lstm = df_lstm.set_index('datetime').sort_index()

feature_cols_linear = [c for c in df_linear.columns if c not in [target_col, 'target_24h_avg']]
feature_cols_lstm = [c for c in df_lstm.columns if c not in [target_col, 'target_24h_avg']]

X_linear = df_linear[feature_cols_linear].values
y_linear = df_linear[target_col].values.astype(int)

X_lstm = df_lstm[feature_cols_lstm].values
y_lstm = df_lstm[target_col].values.astype(int)

print(f"Linear Features ({len(feature_cols_linear)}): {feature_cols_linear[:5]}... (showing first 5)")
print(f"LSTM Features ({len(feature_cols_lstm)}): {feature_cols_lstm[:5]}... (showing first 5)")
print(f"Target: {target_col}")
print(f"Linear: X={X_linear.shape}, y={y_linear.shape}")
print(f"LSTM:   X={X_lstm.shape}, y={y_lstm.shape}")
print(f"Class distribution (Linear): {np.bincount(y_linear)}")
print(f"Class distribution (LSTM): {np.bincount(y_lstm)}")

wandb.log({
    "linear_num_features": len(feature_cols_linear),
    "lstm_num_features": len(feature_cols_lstm),
})

# Split
split_idx_linear = df_linear.index.searchsorted(TEST_SPLIT_DATE)
split_idx_lstm = df_lstm.index.searchsorted(TEST_SPLIT_DATE)

X_train_linear, X_test_linear = X_linear[:split_idx_linear], X_linear[split_idx_linear:]
y_train_linear, y_test_linear = y_linear[:split_idx_linear], y_linear[split_idx_linear:]

X_train_lstm_raw, X_test_lstm_raw = X_lstm[:split_idx_lstm], X_lstm[split_idx_lstm:]
y_train_lstm, y_test_lstm = y_lstm[:split_idx_lstm], y_lstm[split_idx_lstm:]

print(f"\nLinear Train: {len(X_train_linear)} | Test: {len(X_test_linear)}")
print(f"LSTM Train (before windowing): {len(X_train_lstm_raw)} | Test: {len(X_test_lstm_raw)}")

# Scale (features only, NOT targets)
scaler_X_linear = MinMaxScaler()
scaler_X_lstm = MinMaxScaler()

X_train_scaled = scaler_X_linear.fit_transform(X_train_linear)
X_test_scaled = scaler_X_linear.transform(X_test_linear)

X_train_lstm_scaled = scaler_X_lstm.fit_transform(X_train_lstm_raw)
X_test_lstm_scaled = scaler_X_lstm.transform(X_test_lstm_raw)

# Sliding windows
print(f"\nCreating sliding windows (window_size={LSTM_WINDOW_SIZE})...")
X_train_lstm, y_train_lstm_windowed = create_sliding_windows(X_train_lstm_scaled, y_train_lstm, LSTM_WINDOW_SIZE)
X_test_lstm, y_test_lstm_windowed = create_sliding_windows(X_test_lstm_scaled, y_test_lstm, LSTM_WINDOW_SIZE)

print(f"LSTM Train: {X_train_lstm.shape} → targets: {y_train_lstm_windowed.shape}")
print(f"LSTM Test:  {X_test_lstm.shape} → targets: {y_test_lstm_windowed.shape}")

# Convert LSTM targets to categorical for neural networks
y_train_lstm_categorical = to_categorical(y_train_lstm_windowed, num_classes=NUM_CLASSES)
y_test_lstm_categorical = to_categorical(y_test_lstm_windowed, num_classes=NUM_CLASSES)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: eroz (eroz-fpt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Linear Features (45): ['value', 'lag_1h', 'lag_2h', 'lag_3h', 'lag_6h']... (showing first 5)
LSTM Features (23): ['value', 'roll_mean_6h', 'roll_std_6h', 'roll_mean_12h', 'roll_std_12h']... (showing first 5)
Target: target_category
Linear: X=(37641, 45), y=(37641,)
LSTM:   X=(37641, 23), y=(37641,)
Class distribution (Linear): [ 2838 28566  6237]
Class distribution (LSTM): [ 2838 28566  6237]

Linear Train: 31297 | Test: 6344
LSTM Train (before windowing): 31297 | Test: 6344

Creating sliding windows (window_size=72)...
LSTM Train: (31225, 72, 23) → targets: (31225,)
LSTM Test:  (6272, 72, 23) → targets: (6272,)


In [4]:
# 2. MODEL 1: LOGISTIC REGRESSION (baseline) with class weights
from sklearn.utils.class_weight import compute_class_weight

print("[LogisticRegression] Training model with class weights...")

# Compute class weights to handle imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_linear), y=y_train_linear)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class weights: {class_weight_dict}")

lr_model = LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train_linear)

y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_lr_proba = lr_model.predict_proba(X_test_scaled)

lr_f1_weighted = f1_score(y_test_linear, y_pred_lr, average='weighted')
lr_f1_macro = f1_score(y_test_linear, y_pred_lr, average='macro')
lr_balanced_acc = balanced_accuracy_score(y_test_linear, y_pred_lr)
lr_roc_auc = roc_auc_score(y_test_linear, y_pred_lr_proba, multi_class='ovr', average='weighted')

wandb.log({"LogisticRegression/F1-Weighted": lr_f1_weighted, "LogisticRegression/F1-Macro": lr_f1_macro, "LogisticRegression/BalancedAcc": lr_balanced_acc, "LogisticRegression/ROC-AUC": lr_roc_auc})
print(f"[LogisticRegression] F1-Weighted: {lr_f1_weighted:.4f} | F1-Macro: {lr_f1_macro:.4f} | Balanced Acc: {lr_balanced_acc:.4f} | ROC-AUC: {lr_roc_auc:.4f}")

[LogisticRegression] Training model with class weights...
Class weights: {0: np.float64(4.03571889103804), 1: np.float64(0.4488569543642257), 2: np.float64(1.9071907373552712)}
[LogisticRegression] F1-Weighted: 0.7448 | F1-Macro: 0.5402 | Balanced Acc: 0.7230 | ROC-AUC: 0.8177


In [5]:
# 2.5 MODEL 1.5: XGBoost Classifier with sample weights
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

print("[XGBoost] Training model with sample weights...")

val_split = int(len(X_train_linear) * 0.9)
X_train_xgb, X_val_xgb = X_train_linear[:val_split], X_train_linear[val_split:]
y_train_xgb, y_val_xgb = y_train_linear[:val_split], y_train_linear[val_split:]

# Compute sample weights for training data only
sample_weight_train = compute_sample_weight('balanced', y_train_xgb)

xgb_model = XGBClassifier(
    n_estimators=3000, max_depth=7, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.6, colsample_bylevel=0.6,
    min_child_weight=3, reg_alpha=0.05, reg_lambda=1.0, gamma=0.05,
    random_state=SEED, n_jobs=-1, early_stopping_rounds=50, tree_method='hist',
    objective='multi:softprob', num_class=NUM_CLASSES,
)

xgb_model.fit(
    X_train_xgb, y_train_xgb,
    sample_weight=sample_weight_train,
    eval_set=[(X_val_xgb, y_val_xgb)],
    verbose=False,
)

y_pred_xgb = xgb_model.predict(X_test_linear)
y_pred_xgb_proba = xgb_model.predict_proba(X_test_linear)

xgb_f1_weighted = f1_score(y_test_linear, y_pred_xgb, average='weighted')
xgb_f1_macro = f1_score(y_test_linear, y_pred_xgb, average='macro')
xgb_balanced_acc = balanced_accuracy_score(y_test_linear, y_pred_xgb)
xgb_roc_auc = roc_auc_score(y_test_linear, y_pred_xgb_proba, multi_class='ovr', average='weighted')

wandb.log({"XGBoost/F1-Weighted": xgb_f1_weighted, "XGBoost/F1-Macro": xgb_f1_macro, "XGBoost/BalancedAcc": xgb_balanced_acc, "XGBoost/ROC-AUC": xgb_roc_auc})
print(f"[XGBoost] F1-Weighted: {xgb_f1_weighted:.4f} | F1-Macro: {xgb_f1_macro:.4f} | Balanced Acc: {xgb_balanced_acc:.4f} | ROC-AUC: {xgb_roc_auc:.4f}")
print(f"[XGBoost] Best iteration: {xgb_model.best_iteration}")

feat_imp = pd.Series(xgb_model.feature_importances_, index=feature_cols_linear)
print(f"\n[XGBoost] Top 10 features:\n{feat_imp.nlargest(10).to_string()}")

[XGBoost] Training model with sample weights...
[XGBoost] F1-Weighted: 0.8396 | F1-Macro: 0.6353 | Balanced Acc: 0.7015 | ROC-AUC: 0.8335
[XGBoost] Best iteration: 1048

[XGBoost] Top 10 features:
ewm_mean_12h      0.187397
ewm_mean_24h      0.130193
roll_mean_12h     0.062529
roll_mean_24h     0.055373
roll_mean_6h      0.048279
value             0.033164
roll_min_24h      0.031770
month_cos         0.021768
roll_max_24h      0.019890
wind_speed_10m    0.019729


In [6]:
# 2.7 MODEL 1.7: LightGBM Classifier with sample weights
from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.utils.class_weight import compute_sample_weight

print("[LightGBM] Training model with sample weights...")

# Compute sample weights for training data
sample_weight_train_lgb = compute_sample_weight('balanced', y_train_xgb)

lgb_model = LGBMClassifier(
    n_estimators=3000, max_depth=7, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.6, min_child_samples=10,
    reg_alpha=0.05, reg_lambda=1.0, num_leaves=63,
    random_state=SEED, n_jobs=-1, verbose=-1,
    objective='multiclass', num_class=NUM_CLASSES,
)

lgb_model.fit(
    X_train_xgb, y_train_xgb,
    sample_weight=sample_weight_train_lgb,
    eval_set=[(X_val_xgb, y_val_xgb)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
)

y_pred_lgb = lgb_model.predict(X_test_linear)
y_pred_lgb_proba = lgb_model.predict_proba(X_test_linear)

lgb_f1_weighted = f1_score(y_test_linear, y_pred_lgb, average='weighted')
lgb_f1_macro = f1_score(y_test_linear, y_pred_lgb, average='macro')
lgb_balanced_acc = balanced_accuracy_score(y_test_linear, y_pred_lgb)
lgb_roc_auc = roc_auc_score(y_test_linear, y_pred_lgb_proba, multi_class='ovr', average='weighted')

wandb.log({"LightGBM/F1-Weighted": lgb_f1_weighted, "LightGBM/F1-Macro": lgb_f1_macro, "LightGBM/BalancedAcc": lgb_balanced_acc, "LightGBM/ROC-AUC": lgb_roc_auc})
print(f"[LightGBM] F1-Weighted: {lgb_f1_weighted:.4f} | F1-Macro: {lgb_f1_macro:.4f} | Balanced Acc: {lgb_balanced_acc:.4f} | ROC-AUC: {lgb_roc_auc:.4f}")
print(f"[LightGBM] Best iteration: {lgb_model.best_iteration_}")

[LightGBM] Training model with sample weights...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] F1-Weighted: 0.8353 | F1-Macro: 0.6286 | Balanced Acc: 0.7040 | ROC-AUC: 0.8307
[LightGBM] Best iteration: 814


In [7]:
# 3. MODEL 2: LSTM (2-Layer, 128 → 64) for Classification with class weights
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("[LSTM] Building model...")
lstm_model = Sequential([
    Input(shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    LSTM(128, activation='tanh', return_sequences=True),
    Dropout(0.1),
    LSTM(64, activation='tanh'),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
])

lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
print(lstm_model.summary())

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

# Compute class weights for Keras
from sklearn.utils.class_weight import compute_class_weight
lstm_class_weight = compute_class_weight('balanced', classes=np.unique(y_train_lstm), y=y_train_lstm)
lstm_class_weight_dict = {i: w for i, w in enumerate(lstm_class_weight)}
print(f"LSTM Class weights: {lstm_class_weight_dict}")

from tensorflow.keras.callbacks import Callback
class WandbCallback(Callback):
    def __init__(self, model_name):
        self.model_name = model_name
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        for key, value in logs.items():
            wandb.log({f"{self.model_name}/{key}": value})

print("[LSTM] Training (100 epochs, patience=15) with balanced class weights...")
lstm_history = lstm_model.fit(
    X_train_lstm, y_train_lstm_categorical,
    epochs=100, batch_size=32, verbose=1,
    validation_split=0.1, shuffle=False,
    class_weight=lstm_class_weight_dict,
    callbacks=[WandbCallback("LSTM"), early_stop, reduce_lr]
)

y_pred_lstm_proba = lstm_model.predict(X_test_lstm, verbose=0)
y_pred_lstm = np.argmax(y_pred_lstm_proba, axis=1)

lstm_f1_weighted = f1_score(y_test_lstm_windowed, y_pred_lstm, average='weighted')
lstm_f1_macro = f1_score(y_test_lstm_windowed, y_pred_lstm, average='macro')
lstm_balanced_acc = balanced_accuracy_score(y_test_lstm_windowed, y_pred_lstm)
lstm_roc_auc = roc_auc_score(y_test_lstm_windowed, y_pred_lstm_proba, multi_class='ovr', average='weighted')

wandb.log({"LSTM/F1-Weighted": lstm_f1_weighted, "LSTM/F1-Macro": lstm_f1_macro, "LSTM/BalancedAcc": lstm_balanced_acc, "LSTM/ROC-AUC": lstm_roc_auc})
print(f"\n[LSTM] F1-Weighted: {lstm_f1_weighted:.4f} | F1-Macro: {lstm_f1_macro:.4f} | Balanced Acc: {lstm_balanced_acc:.4f} | ROC-AUC: {lstm_roc_auc:.4f}")

[LSTM] Building model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 72, 128)        │        77,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 72, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 129,411 (505.51 KB)

 Trainable params: 129,411 (505.51 KB)

 Non-trainable params: 0 (0.00 B)

None
LSTM Class weights: {0: np.float64(4.03571889103804), 1: np.float64(0.4488569543642257), 2: np.float64(1.9071907373552712)}
[LSTM] Training (100 epochs, patience=15) with balanced class weights...
Epoch 1/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 18s 14ms/step - accuracy: 0.5473 - loss: 1.0189 - val_accuracy: 0.6417 - val_loss: 1.0665 - learning_rate: 0.0010
Epoch 2/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.4057 - loss: 1.1262 - val_accuracy: 0.6417 - val_loss: 1.0614 - learning_rate: 0.0010
Epoch 3/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.3823 - loss: 1.1357 - val_accuracy: 0.6417 - val_loss: 1.0522 - learning_rate: 0.0010
Epoch 4/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.2078 - loss: 1.1367 - val_accuracy: 0.6417 - val_loss: 0.9926 - learning_rate: 0.0010
Epoch 5/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.4230 - loss: 1.1504 - val_accuracy: 0.6417 - val_loss: 1.0297 - learning_rate: 0.0010
Epoch 6/100
879/879 ━━━

In [8]:
# LSTM Diagnostics
print("="*60)
print("LSTM CLASSIFICATION DIAGNOSTICS")
print("="*60)
print(f"X_test shape:  {X_test_lstm.shape}")
print(f"y_test shape:  {y_test_lstm_windowed.shape}")
print(f"y_pred shape:  {y_pred_lstm_proba.shape}")

print(f"\nClass predictions distribution:")
unique, counts = np.unique(y_pred_lstm, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}: {cnt} ({cnt/len(y_pred_lstm)*100:.1f}%)")

print(f"\nActual class distribution:")
unique, counts = np.unique(y_test_lstm_windowed, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}: {cnt} ({cnt/len(y_test_lstm_windowed)*100:.1f}%)")

print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test_lstm_windowed, y_pred_lstm))
print("="*60)

LSTM CLASSIFICATION DIAGNOSTICS
X_test shape:  (6272, 72, 23)
y_test shape:  (6272,)
y_pred shape:  (6272, 3)

Class predictions distribution:
  Class 0: 2727 (43.5%)
  Class 1: 2856 (45.5%)
  Class 2: 689 (11.0%)

Actual class distribution:
  Class 0: 253 (4.0%)
  Class 1: 5268 (84.0%)
  Class 2: 751 (12.0%)

Confusion Matrix:
[[ 192   57    4]
 [2511 2469  288]
 [  24  330  397]]


In [9]:
# 4. MODEL 3: Bi-LSTM (2-Layer, 128 → 64) for Classification with class weights
print("[BiLSTM] Building model...")
bilstm_model = Sequential([
    Input(shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    Bidirectional(LSTM(128, activation='tanh', return_sequences=True)),
    Dropout(0.1),
    Bidirectional(LSTM(64, activation='tanh')),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
])

bilstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
print(bilstm_model.summary())

early_stop_b = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
reduce_lr_b = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

print("[BiLSTM] Training (100 epochs, patience=15) with balanced class weights...")
bilstm_history = bilstm_model.fit(
    X_train_lstm, y_train_lstm_categorical,
    epochs=100, batch_size=32, verbose=1,
    validation_split=0.1, shuffle=False,
    class_weight=lstm_class_weight_dict,
    callbacks=[WandbCallback("BiLSTM"), early_stop_b, reduce_lr_b]
)

y_pred_bilstm_proba = bilstm_model.predict(X_test_lstm, verbose=0)
y_pred_bilstm = np.argmax(y_pred_bilstm_proba, axis=1)

bilstm_f1_weighted = f1_score(y_test_lstm_windowed, y_pred_bilstm, average='weighted')
bilstm_f1_macro = f1_score(y_test_lstm_windowed, y_pred_bilstm, average='macro')
bilstm_balanced_acc = balanced_accuracy_score(y_test_lstm_windowed, y_pred_bilstm)
bilstm_roc_auc = roc_auc_score(y_test_lstm_windowed, y_pred_bilstm_proba, multi_class='ovr', average='weighted')

wandb.log({"BiLSTM/F1-Weighted": bilstm_f1_weighted, "BiLSTM/F1-Macro": bilstm_f1_macro, "BiLSTM/BalancedAcc": bilstm_balanced_acc, "BiLSTM/ROC-AUC": bilstm_roc_auc})
print(f"\n[BiLSTM] F1-Weighted: {bilstm_f1_weighted:.4f} | F1-Macro: {bilstm_f1_macro:.4f} | Balanced Acc: {bilstm_balanced_acc:.4f} | ROC-AUC: {bilstm_roc_auc:.4f}")

[BiLSTM] Building model...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 72, 256)        │       155,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 72, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 324,227 (1.24 MB)

 Trainable params: 324,227 (1.24 MB)

 Non-trainable params: 0 (0.00 B)

None
[BiLSTM] Training (100 epochs, patience=15) with balanced class weights...
Epoch 1/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 21s 20ms/step - accuracy: 0.5286 - loss: 0.9856 - val_accuracy: 0.2395 - val_loss: 1.6365 - learning_rate: 0.0010
Epoch 2/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.4758 - loss: 1.3092 - val_accuracy: 0.2395 - val_loss: 1.5855 - learning_rate: 0.0010
Epoch 3/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 18s 21ms/step - accuracy: 0.4561 - loss: 1.1766 - val_accuracy: 0.2395 - val_loss: 1.0691 - learning_rate: 0.0010
Epoch 4/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.1390 - loss: 1.2130 - val_accuracy: 0.1188 - val_loss: 1.1082 - learning_rate: 0.0010
Epoch 5/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 18s 20ms/step - accuracy: 0.2910 - loss: 1.1042 - val_accuracy: 0.2395 - val_loss: 0.9984 - learning_rate: 0.0010
Epoch 6/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 20s 20ms/step - accuracy: 0.4624 - loss: 1.1991 - val_accuracy: 0.3167 - val_loss: 0.9837 - learning_rat

In [10]:
# 5. ENSEMBLE + VOTING CLASSIFIER

print("[LSTM Ensemble] Averaging LSTM + Bi-LSTM predictions...")
y_pred_ensemble_proba = (y_pred_lstm_proba + y_pred_bilstm_proba) / 2
y_pred_ensemble = np.argmax(y_pred_ensemble_proba, axis=1)

ensemble_f1_weighted = f1_score(y_test_lstm_windowed, y_pred_ensemble, average='weighted')
ensemble_f1_macro = f1_score(y_test_lstm_windowed, y_pred_ensemble, average='macro')
ensemble_balanced_acc = balanced_accuracy_score(y_test_lstm_windowed, y_pred_ensemble)
ensemble_roc_auc = roc_auc_score(y_test_lstm_windowed, y_pred_ensemble_proba, multi_class='ovr', average='weighted')

wandb.log({"Ensemble/F1-Weighted": ensemble_f1_weighted, "Ensemble/F1-Macro": ensemble_f1_macro, "Ensemble/BalancedAcc": ensemble_balanced_acc, "Ensemble/ROC-AUC": ensemble_roc_auc})
print(f"[LSTM Ensemble] F1-Weighted: {ensemble_f1_weighted:.4f} | F1-Macro: {ensemble_f1_macro:.4f} | Balanced Acc: {ensemble_balanced_acc:.4f} | ROC-AUC: {ensemble_roc_auc:.4f}")

# --- Random Forest Classifier ---
print("\n[Random Forest] Training model with balanced class weights...")
rf_model = RandomForestClassifier(n_estimators=500, max_depth=15, random_state=SEED, n_jobs=-1, class_weight='balanced', verbose=1)
rf_model.fit(X_train_linear, y_train_linear)

y_pred_rf = rf_model.predict(X_test_linear)
y_pred_rf_proba = rf_model.predict_proba(X_test_linear)

rf_f1_weighted = f1_score(y_test_linear, y_pred_rf, average='weighted')
rf_f1_macro = f1_score(y_test_linear, y_pred_rf, average='macro')
rf_balanced_acc = balanced_accuracy_score(y_test_linear, y_pred_rf)
rf_roc_auc = roc_auc_score(y_test_linear, y_pred_rf_proba, multi_class='ovr', average='weighted')

wandb.log({"RandomForest/F1-Weighted": rf_f1_weighted, "RandomForest/F1-Macro": rf_f1_macro, "RandomForest/BalancedAcc": rf_balanced_acc, "RandomForest/ROC-AUC": rf_roc_auc})
print(f"[Random Forest] F1-Weighted: {rf_f1_weighted:.4f} | F1-Macro: {rf_f1_macro:.4f} | Balanced Acc: {rf_balanced_acc:.4f} | ROC-AUC: {rf_roc_auc:.4f}")

feat_imp_rf = pd.Series(rf_model.feature_importances_, index=feature_cols_linear)
print(f"\n[Random Forest] Top 10 features:\n{feat_imp_rf.nlargest(10).to_string()}")

# --- MANUAL Soft Voting (average probabilities from 4 models) ---
print("\n[Voting - Manual Soft Voting] Averaging XGB + LGB + RF + LR predictions...")
y_pred_vote_proba = (y_pred_xgb_proba + y_pred_lgb_proba + y_pred_rf_proba + y_pred_lr_proba) / 4
y_pred_vote = np.argmax(y_pred_vote_proba, axis=1)

vote_f1_weighted = f1_score(y_test_linear, y_pred_vote, average='weighted')
vote_f1_macro = f1_score(y_test_linear, y_pred_vote, average='macro')
vote_balanced_acc = balanced_accuracy_score(y_test_linear, y_pred_vote)
vote_roc_auc = roc_auc_score(y_test_linear, y_pred_vote_proba, multi_class='ovr', average='weighted')

wandb.log({"VotingClassifier/F1-Weighted": vote_f1_weighted, "VotingClassifier/F1-Macro": vote_f1_macro, "VotingClassifier/BalancedAcc": vote_balanced_acc, "VotingClassifier/ROC-AUC": vote_roc_auc})
print(f"[Voting - Manual Soft] F1-Weighted: {vote_f1_weighted:.4f} | F1-Macro: {vote_f1_macro:.4f} | Balanced Acc: {vote_balanced_acc:.4f} | ROC-AUC: {vote_roc_auc:.4f}")

[LSTM Ensemble] Averaging LSTM + Bi-LSTM predictions...
[LSTM Ensemble] F1-Weighted: 0.5975 | F1-Macro: 0.4268 | Balanced Acc: 0.5748 | ROC-AUC: 0.5686

[Random Forest] Training model with balanced class weights...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:    3.9s
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:   19.3s
[Parallel(n_jobs=-1)]: Done 446 tasks      | elapsed:   43.2s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:   49.9s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 446 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 500 out of 500 | elapsed:    0.3s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.1s


[Random Forest] F1-Weighted: 0.8478 | F1-Macro: 0.6033 | Balanced Acc: 0.5840 | ROC-AUC: 0.8379

[Random Forest] Top 10 features:
ewm_mean_24h     0.102230
ewm_mean_12h     0.095542
roll_mean_24h    0.089741
roll_mean_12h    0.065909
roll_mean_6h     0.052733
roll_min_24h     0.049837
roll_max_24h     0.043459
value            0.036530
roll_std_24h     0.026280
lag_1h           0.025315

[Voting - Manual Soft Voting] Averaging XGB + LGB + RF + LR predictions...
[Voting - Manual Soft] F1-Weighted: 0.8439 | F1-Macro: 0.6426 | Balanced Acc: 0.7024 | ROC-AUC: 0.8396


[Parallel(n_jobs=2)]: Done 446 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 500 out of 500 | elapsed:    0.3s finished


In [11]:
# 6. COMPARISON & EVALUATION

print("\n" + "="*80)
print("FINAL MODEL COMPARISON — AQI 3-Class Classification (with Class Balancing)")
print("="*80)

def get_classification_metrics(y_true, y_pred, y_proba, name):
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    f1_macro = f1_score(y_true, y_pred, average='macro')
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted')
    acc = np.mean(y_true == y_pred)
    print(f"[{name}]  F1-Mac: {f1_macro:.4f} | BalAcc: {bal_acc:.4f} | F1-W: {f1_weighted:.4f} | Acc: {acc:.4f}")
    return f1_weighted, f1_macro, bal_acc, roc_auc

lr_f1w, lr_f1m, lr_bal_acc, lr_roc_auc = get_classification_metrics(y_test_linear, y_pred_lr, y_pred_lr_proba, "LogisticRegression")
xgb_f1w, xgb_f1m, xgb_bal_acc, xgb_roc_auc = get_classification_metrics(y_test_linear, y_pred_xgb, y_pred_xgb_proba, "XGBoost")
lgb_f1w, lgb_f1m, lgb_bal_acc, lgb_roc_auc = get_classification_metrics(y_test_linear, y_pred_lgb, y_pred_lgb_proba, "LightGBM")
rf_f1w, rf_f1m, rf_bal_acc, rf_roc_auc = get_classification_metrics(y_test_linear, y_pred_rf, y_pred_rf_proba, "RandomForest")
vote_f1w, vote_f1m, vote_bal_acc, vote_roc_auc = get_classification_metrics(y_test_linear, y_pred_vote, y_pred_vote_proba, "VotingClassifier")
lstm_f1w, lstm_f1m, lstm_bal_acc, lstm_roc_auc = get_classification_metrics(y_test_lstm_windowed, y_pred_lstm, y_pred_lstm_proba, "LSTM")
bilstm_f1w, bilstm_f1m, bilstm_bal_acc, bilstm_roc_auc = get_classification_metrics(y_test_lstm_windowed, y_pred_bilstm, y_pred_bilstm_proba, "BiLSTM")
ensemble_f1w, ensemble_f1m, ensemble_bal_acc, ensemble_roc_auc = get_classification_metrics(y_test_lstm_windowed, y_pred_ensemble, y_pred_ensemble_proba, "LSTM-Ensemble")

comparison_data = {
    "Model": ["LogisticRegression", "XGBoost", "LightGBM", "RandomForest", "VotingClassifier", "LSTM", "BiLSTM", "LSTM-Ensemble"],
    "F1-Macro": [lr_f1m, xgb_f1m, lgb_f1m, rf_f1m, vote_f1m, lstm_f1m, bilstm_f1m, ensemble_f1m],
    "F1-Weighted": [lr_f1w, xgb_f1w, lgb_f1w, rf_f1w, vote_f1w, lstm_f1w, bilstm_f1w, ensemble_f1w],
    "Balanced Acc": [lr_bal_acc, xgb_bal_acc, lgb_bal_acc, rf_bal_acc, vote_bal_acc, lstm_bal_acc, bilstm_bal_acc, ensemble_bal_acc],
    "ROC-AUC": [lr_roc_auc, xgb_roc_auc, lgb_roc_auc, rf_roc_auc, vote_roc_auc, lstm_roc_auc, bilstm_roc_auc, ensemble_roc_auc],
}
comparison_df = pd.DataFrame(comparison_data)
print("\n")
print(comparison_df.to_string(index=False))

wandb.log({"model_comparison_table": wandb.Table(dataframe=comparison_df)})

# Best model (by Macro F1 - true measure for imbalanced data)
all_f1_macro = {
    "LogisticRegression": lr_f1m, "XGBoost": xgb_f1m,
    "LightGBM": lgb_f1m, "RandomForest": rf_f1m, "VotingClassifier": vote_f1m,
    "LSTM": lstm_f1m, "BiLSTM": bilstm_f1m, "LSTM-Ensemble": ensemble_f1m,
}
best_model_name = max(all_f1_macro, key=all_f1_macro.get)
best_f1_macro = all_f1_macro[best_model_name]

print(f"\n{'='*80}")
print(f"BEST MODEL: {best_model_name} (F1-Macro = {best_f1_macro:.4f})")
print(f"Using F1-Macro instead of F1-Weighted due to severe class imbalance (21x)")
print(f"{'='*80}\n")

wandb.log({"best_model": best_model_name, "best_f1_macro_score": best_f1_macro})


FINAL MODEL COMPARISON — AQI 3-Class Classification (with Class Balancing)
[LogisticRegression]  F1-Mac: 0.5402 | BalAcc: 0.7230 | F1-W: 0.7448 | Acc: 0.6964
[XGBoost]  F1-Mac: 0.6353 | BalAcc: 0.7015 | F1-W: 0.8396 | Acc: 0.8276
[LightGBM]  F1-Mac: 0.6286 | BalAcc: 0.7040 | F1-W: 0.8353 | Acc: 0.8212
[RandomForest]  F1-Mac: 0.6033 | BalAcc: 0.5840 | F1-W: 0.8478 | Acc: 0.8534
[VotingClassifier]  F1-Mac: 0.6426 | BalAcc: 0.7024 | F1-W: 0.8439 | Acc: 0.8337
[LSTM]  F1-Mac: 0.4294 | BalAcc: 0.5854 | F1-W: 0.5817 | Acc: 0.4876
[BiLSTM]  F1-Mac: 0.3495 | BalAcc: 0.5016 | F1-W: 0.5858 | Acc: 0.5118
[LSTM-Ensemble]  F1-Mac: 0.4268 | BalAcc: 0.5748 | F1-W: 0.5975 | Acc: 0.5056


             Model  F1-Macro  F1-Weighted  Balanced Acc  ROC-AUC
LogisticRegression  0.540192     0.744814      0.722986 0.817729
           XGBoost  0.635325     0.839569      0.701495 0.833544
          LightGBM  0.628643     0.835272      0.704010 0.830720
      RandomForest  0.603301     0.847831      0.583969 0.

In [12]:
# Final detailed results
import os
import json
import joblib

print("\n" + "="*80)
print("DETAILED MODEL PERFORMANCE - AQI 3-Class Classification (Balanced Training)")
print("="*80)
print("\nNote: Class imbalance ratio = 21x (Moderate:Good)")
print("      Using F1-Macro as primary metric (treats all classes equally)\n")

def grade_f1(f1):
    return 'EXCELLENT' if f1 > 0.75 else 'GOOD' if f1 > 0.65 else 'FAIR' if f1 > 0.50 else 'POOR'

models_info = [
    ("LOGISTIC REGRESSION", lr_f1m, lr_bal_acc, lr_roc_auc),
    ("XGBOOST", xgb_f1m, xgb_bal_acc, xgb_roc_auc),
    ("LIGHTGBM", lgb_f1m, lgb_bal_acc, lgb_roc_auc),
    ("RANDOM FOREST", rf_f1m, rf_bal_acc, rf_roc_auc),
    ("VOTING CLASSIFIER", vote_f1m, vote_bal_acc, vote_roc_auc),
    ("LSTM", lstm_f1m, lstm_bal_acc, lstm_roc_auc),
    ("BI-LSTM", bilstm_f1m, bilstm_bal_acc, bilstm_roc_auc),
    ("LSTM-ENSEMBLE", ensemble_f1m, ensemble_bal_acc, ensemble_roc_auc),
]

for name, f1m, bal_acc, roc_auc in models_info:
    print(f"\n{name}")
    print(f"   F1-Macro: {f1m:.4f} ({grade_f1(f1m)}) | BalAcc: {bal_acc:.4f} | ROC-AUC: {roc_auc:.4f}")

print("\n" + "="*80)
print(f"BEST MODEL: {best_model_name} (F1-Macro = {best_f1_macro:.4f}, {grade_f1(best_f1_macro)})")
print("="*80)

print("\nFull Comparison Table (sorted by F1-Macro):")
comparison_df_sorted = comparison_df.sort_values('F1-Macro', ascending=False)
print(comparison_df_sorted.to_string(index=False))

# Classification reports for best model
print("\n\n" + "="*80)
print(f"CLASSIFICATION REPORT - {best_model_name} (Best Model)")
print("="*80)

if best_model_name in ["VotingClassifier", "XGBoost", "LightGBM", "RandomForest", "LogisticRegression"]:
    if best_model_name == "VotingClassifier":
        y_pred_best = y_pred_vote
        y_test_best = y_test_linear
    elif best_model_name == "XGBoost":
        y_pred_best = y_pred_xgb
        y_test_best = y_test_linear
    elif best_model_name == "LightGBM":
        y_pred_best = y_pred_lgb
        y_test_best = y_test_linear
    elif best_model_name == "RandomForest":
        y_pred_best = y_pred_rf
        y_test_best = y_test_linear
    else:  # LogisticRegression
        y_pred_best = y_pred_lr
        y_test_best = y_test_linear
else:  # LSTM-based
    if best_model_name == "LSTM":
        y_pred_best = y_pred_lstm
    elif best_model_name == "BiLSTM":
        y_pred_best = y_pred_bilstm
    else:  # LSTM-Ensemble
        y_pred_best = y_pred_ensemble
    y_test_best = y_test_lstm_windowed

print(classification_report(y_test_best, y_pred_best, target_names=['Good', 'Moderate', 'Poor']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_best, y_pred_best))

# --- SAVE MODELS FOR WEB DEMO ---
os.makedirs("artifacts", exist_ok=True)

# Save tabular models + scaler + feature list
joblib.dump(lr_model, "artifacts/logistic_regression.pkl")
joblib.dump(xgb_model, "artifacts/xgboost.pkl")
joblib.dump(lgb_model, "artifacts/lightgbm.pkl")
joblib.dump(rf_model, "artifacts/random_forest.pkl")
joblib.dump(scaler_X_linear, "artifacts/scaler_X_linear.pkl")
joblib.dump(feature_cols_linear, "artifacts/feature_cols_linear.pkl")

# Save deep learning models
lstm_model.save("artifacts/lstm_model.keras")
bilstm_model.save("artifacts/bilstm_model.keras")

# Save metadata for deployment
metadata = {
    "best_model_name": best_model_name,
    "best_f1_macro": float(best_f1_macro),
    "class_names": ["Good", "Moderate", "Poor"],
    "target_mapping": {"0": "Good", "1": "Moderate", "2": "Poor"},
    "linear_models": [
        "logistic_regression.pkl",
        "xgboost.pkl",
        "lightgbm.pkl",
        "random_forest.pkl"
    ],
    "deep_models": [
        "lstm_model.keras",
        "bilstm_model.keras"
    ],
    "preprocessing": {
        "scaler": "scaler_X_linear.pkl",
        "feature_columns": "feature_cols_linear.pkl"
    },
    "manual_soft_voting": {
        "enabled": True,
        "formula": "(xgb_proba + lgb_proba + rf_proba + lr_proba) / 4"
    }
}

with open("artifacts/model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\nSaved artifacts:")
print("- artifacts/logistic_regression.pkl")
print("- artifacts/xgboost.pkl")
print("- artifacts/lightgbm.pkl")
print("- artifacts/random_forest.pkl")
print("- artifacts/scaler_X_linear.pkl")
print("- artifacts/feature_cols_linear.pkl")
print("- artifacts/lstm_model.keras")
print("- artifacts/bilstm_model.keras")
print("- artifacts/model_metadata.json")

wandb.finish()


DETAILED MODEL PERFORMANCE - AQI 3-Class Classification (Balanced Training)

Note: Class imbalance ratio = 21x (Moderate:Good)
      Using F1-Macro as primary metric (treats all classes equally)


LOGISTIC REGRESSION
   F1-Macro: 0.5402 (FAIR) | BalAcc: 0.7230 | ROC-AUC: 0.8177

XGBOOST
   F1-Macro: 0.6353 (FAIR) | BalAcc: 0.7015 | ROC-AUC: 0.8335

LIGHTGBM
   F1-Macro: 0.6286 (FAIR) | BalAcc: 0.7040 | ROC-AUC: 0.8307

RANDOM FOREST
   F1-Macro: 0.6033 (FAIR) | BalAcc: 0.5840 | ROC-AUC: 0.8379

VOTING CLASSIFIER
   F1-Macro: 0.6426 (FAIR) | BalAcc: 0.7024 | ROC-AUC: 0.8396

LSTM
   F1-Macro: 0.4294 (POOR) | BalAcc: 0.5854 | ROC-AUC: 0.6322

BI-LSTM
   F1-Macro: 0.3495 (POOR) | BalAcc: 0.5016 | ROC-AUC: 0.5291

LSTM-ENSEMBLE
   F1-Macro: 0.4268 (POOR) | BalAcc: 0.5748 | ROC-AUC: 0.5686

BEST MODEL: VotingClassifier (F1-Macro = 0.6426, FAIR)

Full Comparison Table (sorted by F1-Macro):
             Model  F1-Macro  F1-Weighted  Balanced Acc  ROC-AUC
  VotingClassifier  0.642586     0.84

BiLSTM/BalancedAcc,▁
BiLSTM/F1-Macro,▁
BiLSTM/F1-Weighted,▁
BiLSTM/ROC-AUC,▁
BiLSTM/accuracy,▆▆▅▁▄▅▅▆▇▇▇█████████████
BiLSTM/loss,▇█▇█▇▇▄▂▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
BiLSTM/val_accuracy,▂▂▂▁▂▃▇▆███▇▇▇▇▇▇▇▇▇▇█▇█
BiLSTM/val_loss,██▄▄▃▃▁▂▁▁▁▁▁▂▁▂▂▂▂▁▁▁▁▁
Ensemble/BalancedAcc,▁
Ensemble/F1-Macro,▁
+33,...
